# 01 — EDA: laps, tyres, telemetry (Day 3)

Exploration of the Prediction-lane tables for the 8 races in `configs/races.toml`,
to decide what the tyre model (Days 4–5) and lap-time model (Days 6–7) train on.

Regenerate the data first if `data/processed/` is empty: `python -m src.ingestion.run_ingestion`.

**Leakage note.** This notebook looks at whole races on purpose — that is fine for EDA and for
choosing *row filters* (which only look at the row itself). Anything that becomes a model
*feature* must be computed through `src.preprocessing.leakage.as_of_lap()`.

In [ ]:
import os, sys
from pathlib import Path

# Run from the repo root so `src` imports and data/ paths resolve (Jupyter starts in notebooks/).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.ingestion.storage import connect
from src.preprocessing.leakage import as_of_lap

pd.set_option("display.width", 160)
con = connect()
con.sql("SHOW TABLES").df()

## 1. Coverage

In [ ]:
con.sql("SELECT race_id, circuit, total_laps, weather_summary FROM races ORDER BY round").df()

In [ ]:
con.sql('''
    SELECT race_id,
           COUNT(*)                            AS laps,
           COUNT(DISTINCT driver)              AS drivers,
           AVG((lap_time IS NULL)::INT)        AS null_lap_time,
           AVG(is_accurate::INT)               AS accurate,
           SUM(pit_flag::INT)                  AS pit_in_laps,
           COUNT(*) FILTER (WHERE regexp_matches(track_status, '[4567]')) AS neutralised_laps,
           MAX(gap_to_leader)                  AS max_gap
    FROM laps GROUP BY race_id ORDER BY race_id
''').df().round(3)

- ~9.2k driver-laps. Italy and Singapore have 19 drivers (DNS/early DNF), the rest 20.
- Null `lap_time` is concentrated where the race was stopped: Australia (102 — three red flags),
  Netherlands (33), Singapore (26). Elsewhere ≤ 9.
- Monaco, Spain and Italy have **no** SC/VSC/red-flag laps; Australia, Netherlands, Britain and
  Singapore carry nearly all of them.

## 2. One joined frame

In [ ]:
df = con.sql('''
    SELECT l.*, t.stint, t.compound, t.tyre_age_at_lap, t.fresh_tyre, r.circuit
    FROM laps l
    JOIN tyres t USING (race_id, driver, lap_number)
    JOIN races r USING (race_id)
''').df()

neutralised = df["track_status"].fillna("").str.contains("[4567]")
df["clean"] = (
    (df["lap_number"] > 1)          # standing start
    & ~df["pit_flag"] & ~df["pit_out_flag"]
    & ~neutralised                  # SC / VSC / red flag
    & df["lap_time"].notna()
    & df["is_accurate"]
)
print(f"{len(df)} laps, {df['clean'].mean():.1%} clean")

## 3. Tyre compounds

In [ ]:
clean = df[df["clean"]].copy()
clean.groupby(["race_id", "compound"]).size().unstack(fill_value=0)

- Dry compounds dominate. INTERMEDIATE/WET only in Monaco (laps 52–78) and Netherlands (early and
  late rain). 48 WET laps in total — too few to model; drop WET, treat INTERMEDIATE separately or drop.
- Compound usage is very uneven per circuit (Australia is almost all HARD, Italy has no SOFT,
  Bahrain almost no MEDIUM). A per-compound model will have holes per circuit, which argues for
  **compound as a feature** in one model (Design.md §6.3 asks to compare both).
- Max tyre age ~55 laps (HARD/MEDIUM). Ages > 35 are sparse.

## 4. Lap-time outliers among clean laps

In [ ]:
clean["rel_pace"] = clean["lap_time"] / clean.groupby("race_id")["lap_time"].transform("median")
print(clean["rel_pace"].describe(percentiles=[.01, .05, .95, .99]).round(3))

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(clean["rel_pace"].clip(upper=1.4), bins=120)
ax.axvline(1.07, color="k", ls="--", lw=1)
ax.set_xlabel("lap time / race median"); ax.set_ylabel("laps")
plt.show()

- Even "clean" laps have a long slow tail (99th pct = 1.30× median): wet laps, damage, yellow
  sectors not flagged at lap level, traffic.
- A **1.07× race-median** cut removes ~9% of clean laps and leaves a tight dry-pace distribution.
  Note this uses the race median, which is whole-race information: OK as a training-row filter,
  **not** OK as a live feature. At inference the equivalent must be computed `as_of_lap`.

## 5. Degradation signal

Within one stint, tyre age and lap number rise together, so a plain lap-time-vs-age slope mixes
tyre wear with fuel burn-off (≈ −0.05 s/lap) and track evolution. A per-stint regression can't
separate them — they're perfectly collinear.

What does separate them: on the **same lap of the same race**, cars run different tyre ages.
So regress lap time on lap dummies (fuel + track evolution, shared by all cars) + driver dummies
(car/driver pace) + per-compound age slopes and offsets.

In [ ]:
dry = clean[clean["compound"].isin(["SOFT", "MEDIUM", "HARD"]) & (clean["rel_pace"] < 1.07)]

rows = []
for race_id, g in dry.groupby("race_id"):
    X = pd.concat([pd.get_dummies("L" + g["lap_number"].astype(str)),
                   pd.get_dummies(g["driver"])], axis=1).astype(float)
    for comp in ["SOFT", "MEDIUM"]:
        if (g["compound"] == comp).sum() > 30:
            X[f"offset_{comp}"] = (g["compound"] == comp).astype(float)  # vs HARD
            X[f"deg_{comp}"] = g["tyre_age_at_lap"].astype(float) * (g["compound"] == comp)
    if (g["compound"] == "HARD").sum() > 30:
        X["deg_HARD"] = g["tyre_age_at_lap"].astype(float) * (g["compound"] == "HARD")
    beta, *_ = np.linalg.lstsq(X.values, g["lap_time"].values, rcond=None)
    b = dict(zip(X.columns, beta))
    resid = g["lap_time"].values - X.values @ beta
    rows.append({"race_id": race_id, "n": len(g), "resid_sd": resid.std(),
                 **{k: v for k, v in b.items() if k.startswith(("deg_", "offset_"))}})

deg = pd.DataFrame(rows).set_index("race_id")
deg[["n", "deg_SOFT", "deg_MEDIUM", "deg_HARD", "offset_SOFT", "offset_MEDIUM", "resid_sd"]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, comp in zip(axes, ["SOFT", "MEDIUM", "HARD"]):
    for race_id, g in dry[dry["compound"] == comp].groupby("race_id"):
        # Remove each lap's field-average so fuel/track evolution drops out, then average by age.
        g = g.assign(delta=g["lap_time"] - dry.groupby(["race_id", "lap_number"])["lap_time"]
                     .transform("mean").loc[g.index])
        curve = g.groupby("tyre_age_at_lap")["delta"].mean()
        curve = curve[g.groupby("tyre_age_at_lap").size() >= 5]
        ax.plot(curve.index, curve.values, label=race_id.removeprefix("2023_"), lw=1)
    ax.set_title(comp); ax.set_xlabel("tyre age (laps)")
axes[0].set_ylabel("lap time − field mean on that lap (s)")
axes[-1].legend(fontsize=7)
plt.tight_layout(); plt.show()

Degradation (s/lap of tyre age, controlling for lap and driver):

| | SOFT | MEDIUM | HARD |
|---|---|---|---|
| Bahrain | 0.13 | — | 0.10 |
| Singapore | 0.05 | 0.08 | 0.05 |
| Australia | — | 0.08 | 0.01 |
| Italy | — | 0.05 | 0.06 |
| Spain | 0.04 | 0.05 | 0.04 |
| Britain | 0.03 | 0.04 | 0.02 |
| Netherlands | 0.04 | 0.05 | ≈0 |
| Monaco | — | 0.02 | 0.03 |

- Signal is real and plausible: Bahrain is the known high-deg track, Monaco the lowest.
- **Circuit matters more than compound** — within a circuit the three compounds are often close.
  Circuit must be a feature.
- Residual SD 0.34–0.83 s per lap: lap-to-lap noise is large relative to per-lap degradation
  (~0.05 s), so the tyre model should be judged on the *curve* over a stint, not single laps.
- Target definition for Day 4: `pace_loss` = lap time − driver's own early-stint reference pace,
  fuel-corrected with a fixed per-lap coefficient (or lap-effect estimate). The reference must come
  from laps ≤ current lap only.

## 6. `gap_to_leader` on stopped laps

In [ ]:
df[df["gap_to_leader"] > 300].groupby(["race_id", "lap_number"]).agg(
    drivers=("driver", "size"), max_gap=("gap_to_leader", "max"), track_status=("track_status", "first"))

- Gaps of 600–4000 s happen exactly on red-flag laps (Australia L8/18/55/57, Netherlands L15/64):
  the session clock keeps running while the field waits in the pit lane.
- Monaco L68–71 (~300–330 s) is a lapped car in the rain — legitimately large.
- Plan (Day 4 preprocessing): set gap to NaN on laps with track status 5 and on laps where the gap
  jumps implausibly; forward-fill only from earlier laps.

## 7. Telemetry

In [ ]:
tel = con.sql("SELECT * FROM telemetry").df()
print(tel.groupby("race_id")["n_samples"].describe()[["count", "min", "50%", "max"]])
tel.describe().round(2)

- One row per driver-lap with coverage matching the Lap table. `n_samples` median ~290–380 per lap.
- `n_samples` of 12k–15k (Australia, Netherlands) are red-flag laps that include the stoppage;
  their speed/throttle summaries are meaningless — exclude with the same neutralised-lap filter.
- These are per-lap summaries of the lap just driven, so using lap N's telemetry at lap N is fine;
  using lap N+1's to predict lap N+1 is not (the lap-time model must lag them).

## 8. The guard on real data

In [ ]:
bahrain = df[df["race_id"] == "2023_bahrain"]
at_lap_20 = as_of_lap(bahrain, "lap_number", 20)
assert at_lap_20["lap_number"].max() == 20
print(len(bahrain), "rows ->", len(at_lap_20), "rows visible at lap 20")

## Takeaways for Days 4–7

1. **Training rows:** drop lap 1, in/out laps, SC/VSC/red-flag laps, null or inaccurate lap times,
   and laps slower than 1.07× the race median. ~78% of laps survive.
2. **Dry model only:** drop WET; exclude INTERMEDIATE from the tyre model (≈600 laps, 2 races —
   too thin), revisit if time allows.
3. **Features:** compound as a categorical feature (per-compound models have holes by circuit);
   circuit is essential; fuel/lap effect must be removed before measuring degradation.
4. **Evaluate on held-out races** as well as held-out laps: only 8 circuits, and degradation
   varies more between circuits than between compounds.
5. **Track temperature** is needed per lap for `predict_tyre_degradation()` — waiting on the
   partner's WeatherSnapshot table. Race-level `weather_summary` must not be used.
6. **Clean `gap_to_leader`** on red-flag laps before it becomes a feature.